<a href="https://colab.research.google.com/github/Mohamed-AboZahra/L3_Library_Project/blob/main/L3_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import sqlite3, pandas as pd, json
from bs4 import BeautifulSoup

conn = sqlite3.connect("download.db")
for t in ["members", "books", "checkouts"]:
    print(t, pd.read_sql_query(f"SELECT * FROM {t};", conn))

members     member_id first_name last_name  grade neighborhood membership_status  \
0        1001      Salma   Ibrahim    8.0        Maadi            Active   
1        1002      Fares     Saleh    9.0        Maadi            Active   
2        1003     Bassel    Hegazy    6.0        Maadi            Active   
3        1004      Fares     Wahba    7.0        Maadi          inactive   
4        1005    Youssef     Halim    9.0        Maadi            Active   
..        ...        ...       ...    ...          ...               ...   
75       1076       Dina     Wahba    7.0       Shubra            Active   
76       1077       Lina    Rashad    6.0       Shubra            Active   
77       1078     Habiba     Osman    7.0       Shubra          INACTIVE   
78       1079       Rana     Osman    8.0       Shubra            Active   
79       1080     Bassel     Wahba    9.0       Shubra            Active   

     join_date  
0   2023-04-05  
1         None  
2   2025-04-23  
3   2024-10

In [6]:
# Q1: checkouts per member, including zero
q1 = pd.read_sql_query("""
    SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS total_checkouts
    FROM members m
    LEFT JOIN checkouts c ON m.member_id = c.member_id
    GROUP BY m.member_id, m.first_name, m.last_name
    ORDER BY total_checkouts DESC, m.member_id;
""", conn)
print("Q1: checkouts per member, including zero")
print(q1)

# Q2: authors starting with "A"
q2 = pd.read_sql_query("""
    SELECT book_id, title, author FROM books
    WHERE author LIKE 'A%' ORDER BY author, title;
""", conn)
print("Q2: authors starting with 'A'")
print(q2)

# Q3: top 5 most-borrowed titles
q3 = pd.read_sql_query("""
    SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_checked_out
    FROM checkouts c JOIN books b ON c.book_id = b.book_id
    GROUP BY b.book_id ORDER BY times_checked_out DESC LIMIT 5;
""", conn)
print("Q3: top 5 most-borrowed titles")
print(q3)

# Q4: top 10 most active members
q4 = pd.read_sql_query("""
    SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS total_checkouts
    FROM members m JOIN checkouts c ON m.member_id = c.member_id
    GROUP BY m.member_id ORDER BY total_checkouts DESC LIMIT 10;
""", conn)
print("Q4: top 10 most active members")
print(q4)

# Q5: Maadi checkouts, newest->oldest, skipping the 10 most recent
q5 = pd.read_sql_query("""
    SELECT c.checkout_id, m.member_id, m.first_name, b.title, c.checkout_date, c.return_date
    FROM checkouts c
    JOIN members m ON c.member_id = m.member_id
    JOIN books b ON c.book_id = b.book_id
    WHERE TRIM(m.neighborhood) = 'Maadi' COLLATE NOCASE
    ORDER BY c.checkout_date DESC LIMIT -1 OFFSET 10;
""", conn)
print("Q5: Maadi checkouts, newest->oldest, skipping the 10 most recent")
print(q5)

Q1: checkouts per member, including zero
    member_id first_name last_name  total_checkouts
0        1034        Aya     Wahba               25
1        1044     Sherif     Saleh               21
2        1008       Ziad     Saleh               19
3        1010       Nour     Nabil               18
4        1027    Mostafa     Fouad               18
..        ...        ...       ...              ...
75       1063      Layla     Fouad                0
76       1064      Fares     Sabry                0
77       1066       Amir     Wahba                0
78       1069     Bassel      Adel                0
79       1078     Habiba     Osman                0

[80 rows x 4 columns]
Q2: authors starting with 'A'
   book_id                title         author
0      504  Rooftop Astronomers   Adel Roushdy
1      503    The Lantern Maker   Adel Roushdy
2      502       Desert Compass  Amina Darwish
3      501      The Silver Kite  Amina Darwish
4      505  Letters to the Nile      Aya Hafez


In [7]:
# Stage 1-(pandas merge only, no SQL join)
members_df   = pd.read_sql_query("SELECT * FROM members;", conn)
books_df     = pd.read_sql_query("SELECT * FROM books;", conn)
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts;", conn)

stage1 = checkouts_df.merge(members_df, on="member_id", how="left")
stage1["member_total_checkouts"] = stage1.groupby("member_id")["checkout_id"].transform("count")

In [8]:
# Stage 2-(fold in book catalog)
catalog_df = pd.DataFrame(json.load(open("download.json")))
book_details = books_df.merge(catalog_df, on="book_id", how="left")
stage2 = stage1.merge(book_details, on="book_id", how="left")
assert len(stage2) == len(checkouts_df)   # no rows lost/duplicated

In [9]:
# Stage 3-(parse the Reading Kickoff HTML page)
soup = BeautifulSoup(open("download.html").read(), "html.parser")
rows = [[td.get_text(strip=True) for td in tr.find_all("td")]
        for tr in soup.find("table").find_all("tr")[1:]]

rk_df = pd.DataFrame(rows, columns=["member_id","book_id","checkout_date"]).astype({"member_id":int,"book_id":int})
rk_df["checkout_id"] = ["RK-"+str(i+1) for i in range(len(rk_df))]
rk_df["return_date"] = None

rk_full = rk_df.merge(members_df, on="member_id", how="left").merge(book_details, on="book_id", how="left")
rk_full["member_total_checkouts"] = None

combined = pd.concat([stage2, rk_full[stage2.columns]], ignore_index=True)
combined["member_total_checkouts"] = combined.groupby("member_id")["checkout_id"].transform("count")

print(combined.shape, combined.columns.tolist())
combined.to_csv("task1_combined_data.csv", index=False)

(417, 18) ['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'member_total_checkouts', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher']


In [10]:
import pandas as pd
df = pd.read_csv("task1_combined_data.csv")

# Problem 1: missing values — deliberately left as NaN (see report for reasoning per column)

df.to_csv("task2_cleaned_data.csv", index=False)
print(df.shape)

# Problem 2: true duplicates (identical on every column except checkout_id)
cols_no_id = [c for c in df.columns if c != "checkout_id"]
df = df.drop_duplicates(subset=cols_no_id, keep="first")

# Problem 3: standardize inconsistent text
df["neighborhood"] = df["neighborhood"].str.strip().str.title()
df["membership_status"] = df["membership_status"].str.strip().str.title()

# Problem 4: invalid member_id — remove first (independent of the rest)
invalid_ids = {1104, 1150, 1201}
df = df[~df["member_id"].isin(invalid_ids)].copy()

(417, 18)
